# Clipping Videos with Transcript Timestamps using Parakeet.

One of the most common usecases for video ai is converting long form content into clips. For instance, if you regularly host a 2 hour podcast and want to promote your show using short-form content, we can use llms to highlight the most compelling moments of each episode. 

We can then take each of these clips and translate them into different languages to improve localization and accessibility. 


<a target="_blank" href="https://colab.research.google.com/github/everettVT/daft-video-embeddings/blob/main/workload/notebooks/video_clips_from_transcriptions.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


In [ ]:
!pip install "daft[huggingface]" "nemo_toolkit[asr]" av numpy

In [2]:
uri = "/Users/everett-founder/Movies/*.mp4"
B, T, H, W, C = 2, 16, 288, 288, 3 # Batch Size, Clip Size (# frames), Height, Width, RGB
ROW_LIMIT = 500

In [3]:
import daft
import numpy as np
from daft import col, DataType as dt
from daft.functions import file, llm_generate
import av
from av.audio.resampler import AudioResampler

In [ ]:
df = (
    daft.from_glob_path(uri)
    .with_column("file", file(col("path")))
)

In [9]:
@daft.func(return_dtype = dt.struct({
    "width": dt.int32(),
    "height": dt.int32(),
    "fps": dt.float64(),
    "duration": dt.float64(),
    "frame_count": dt.int32(),
    "time_base": dt.float64(),
}))
def fetch_meta(
    file: daft.File,
    *,
    probesize: str = "64k",
    analyzeduration_us: int = 200_000,
) -> dict:
    """
    Extract basic video metadata from container headers.

    Returns
    -------
    dict
        width, height, fps, frame_count, time_base, keyframe_pts, keyframe_indices
    """
    options = {
        "probesize": str(probesize),
        "analyzeduration": str(analyzeduration_us),
    }

    with av.open(file, mode="r", options=options, metadata_encoding="utf-8") as container:
        video = next(
            (stream for stream in container.streams if stream.type == "video"),
            None,
        )
        if video is None:
            return {
                "width": None,
                "height": None,
                "fps": None,
                "frame_count": None,
                "time_base": None,
                "keyframe_pts": [],
                "keyframe_indices": [],
            }

        # Basic stream properties ----------
        width = video.width
        height = video.height
        time_base = float(video.time_base) if video.time_base else None

        # Frame rate -----------------------
        fps = None
        if video.average_rate:
            fps = float(video.average_rate)
        elif video.guessed_rate:
            fps = float(video.guessed_rate)

        # Duration -------------------------
        duration = None
        if container.duration and container.duration > 0:
            duration = container.duration / 1_000_000.0
        elif video.duration:
            # Fallback time_base only for duration computation if missing
            tb_for_dur = float(video.time_base) if video.time_base else (1.0 / 1_000_000.0)
            duration = float(video.duration * tb_for_dur)

        # Frame count -----------------------
        frame_count = video.frames
        if not frame_count or frame_count <= 0:
            if duration and fps:
                frame_count = int(round(duration * fps))
            else:
                frame_count = None

        return {
            "width": width,
            "height": height,
            "fps": fps,
            "duration": duration,
            "frame_count": frame_count,
            "time_base": time_base
        }

In [10]:
df = df.with_column("metadata", fetch_meta(col("file")))

DaftCoreException: DaftError::FieldNotFound Column col(file) not found.

In [ ]:
@daft.func(return_dtype=dt.struct({
    "index": dt.list(dt.uint64()),
    "pts": dt.list(dt.float64()),
}))
def key_frames(
    file: daft.File,
    *,
    probesize: str = "64k",
    analyzeduration_us: int = 200_000,
) -> list[float]:

    options = {
            "probesize": str(probesize),
            "analyzeduration": str(analyzeduration_us),
        }

    with av.open(file,mode="r", options=options, metadata_encoding="utf-8") as container:
        video = next(
            (stream for stream in container.streams if stream.type == "video"),
            None,
        )
        if video is None:
            return {
                "index": [],
                "pts": [],
            }

        fps = None
        if video.average_rate:
            fps = float(video.average_rate)
        elif video.guessed_rate:
            fps = float(video.guessed_rate)

        keyframe_pts = []
        try:
            for packet in container.demux(video):
                if packet.is_keyframe and packet.pts is not None:
                    pts_seconds = float(packet.pts * float(video.time_base))
                    keyframe_pts.append(pts_seconds)
        except Exception:
            keyframe_pts = []

        keyframe_indices = (
            [int(round(t * fps)) for t in keyframe_pts] if fps else []
        )

        return {
            "index": keyframe_indices,
            "pts": keyframe_pts,
        }


In [6]:
df = df.with_column("key_frames", key_frames(col("file"))).collect()
df.show()

DaftCoreException: DaftError::FieldNotFound Column col(file) not found.

In [ ]:
@daft.func()
def seek_audio(file: daft.File, start_sec: float, end_sec: float, num_frames: int = 16, ) -> np.ndarray:

    container = av.open(file)
    resampler = AudioResampler(format='s16', layout='mono', rate=16000)

    chunks = []
    try:
        for frame in container.decode(audio=0):
            # Resample to desired SR/mono/PCM16; result can be a frame or list of frames
            res = resampler.resample(frame)
            frames = res if isinstance(res, (list, tuple)) else [res]

            for f in frames:
                arr = f.to_ndarray()  # typically (channels, samples) or (samples,)

                # Convert PCM16 → float32 in [-1, 1]
                if arr.dtype != np.float32:
                    arr = (arr.astype(np.float32) / 32768.0).clip(-1.0, 1.0)

                chunks.append(arr)
    finally:
        container.close()

    if not chunks:
        return np.zeros((0,), dtype=np.float32)

    audio = np.concatenate(chunks, axis=0).astype(np.float32, copy=False)
    return audio



In [ ]:
df = df.with_column("audio", seek_audio(col("file")))

In [ ]:
# Parakeet Transcribe with Timestamps
@daft.udf(return_dtype = dt.list(dt.struct({
        "start_offset": dt.int32(),
        "end_offset": dt.int32(),
        "start": dt.float32(),
        "end": dt.float32()
    }))
)
class ParakeetTranscribeTimestampsUDF:
    def __init__(self, context_size: int = 256):
        import nemo.collections.asr as nemo_asr
        self.asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v3")
        self.asr_model.change_attention_model(
            self_attention_model="rel_pos_local_attn",
            att_context_size=[context_size, context_size]
        )

    def __call__(self, audio: list[np.ndarray]):
        outputs = self.asr_model.transcribe(audio, timestamps=True)   # No public flag to emit only segments
        return [o.timestamp["segment"] for o in outputs]

In [ ]:
df = df.with_column("transcripts_w_ts", ParakeetTranscribeTimestampsUDF(col("audio")))

Now that we have the keyframes and timestamped transcripts for each video, we'll ask an llm for top moments from each video. 

In [ ]:
from pydantic import BaseModel, Field
import json

class VideoClip(BaseModel):
    notepad: str = Field(..., description="A field for you to deliberate, fill with internal CoT")
    clip_start_sec: float = Field(..., description = "The start timestamp of the clip in seconds as a float")
    clip_end_sec: float = Field(..., description = "The end timestamp of the clip in seconds as a float")
    transcript: str = Field(..., description = "The combined content of the clip's transcript")
    language: str = Field(..., description = "The Langauge of the transcript")

response_model = {
    "type": "json_schema",
    "json_schema": {
        "name": "math-response",
        "schema": VideoClip.model_json_schema(),
    },
}

@daft.func()
def transcript_timestamps_to_json(transcript_segment: list[dict]) -> str:
    return json.dumps(transcript_segment)


In [ ]:
from daft.functions import format
from daft import lit

df = df.with_column("top_moments", 
    llm_generate(
        format("Identify the most compelling 15 second clip from the following transcript: \n {}", col("path")),
        model="gemma-3-270m-it-mlx",
        provider="openai",
        api_key = "none",
        base_url = "http://127.0.0.1:1234"
    )
).collect()

🗡️ 🐟 InMemorySource: 00:00 

🗡️ 🐟 Project: 00:00 

🗡️ 🐟 UDF _OpenAIGenerator: 00:00 

Error when running pipeline node UDF _OpenAIGenerator


RuntimeError: UDF unexpectedly failed with traceback:
Traceback (most recent call last):

  File "/Users/everett-founder/git/dream/daft-video-embeddings/.venv/lib/python3.11/site-packages/daft/functions/llm.py", line 134, in __init__
    from openai import AsyncOpenAI

ModuleNotFoundError: No module named 'openai'


During handling of the above exception, another exception occurred:


Traceback (most recent call last):

  File "/Users/everett-founder/git/dream/daft-video-embeddings/.venv/lib/python3.11/site-packages/daft/execution/udf_worker.py", line 49, in udf_event_loop
    initialized_projection = ExpressionsProjection([e._initialize_udfs() for e in uninitialized_projection])
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/Users/everett-founder/git/dream/daft-video-embeddings/.venv/lib/python3.11/site-packages/daft/execution/udf_worker.py", line 49, in <listcomp>
    initialized_projection = ExpressionsProjection

In [ ]:
from pydantic import BaseModel, Field

class VideoClip(BaseModel):
    start: float
    end: float
    transcript: str
    language: str
    audio: np.ndarray
    video: np.ndarray




NameError: name 'daft' is not defined